# Apply the Llama model 70B online

## Information

Inference API of Hugging Face exposes models that have large community interest and are in active use: https://huggingface.co/docs/api-inference/supported-models


**Precondition**: create an Access Token (https://huggingface.co/settings/tokens), set up a pro account to use the larger LLMs like Llama-3-70B (https://huggingface.co/pricing#pro) and accept the META LLAMA 3 COMMUNITY LICENSE AGREEMENT for the two different Llama models:

* for `meta-llama/Meta-Llama-3-70B-Instruct`: https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct

Remark: Llama models are published under the **META LLAMA 3 COMMUNITY LICENSE AGREEMENT**. The Meta Llama 3 Community License grants users a non-exclusive, royalty-free license (you not need to pay ongoing fees) to use, modify, and distribute Llama 3 materials, with requirements for attribution and naming conventions when creating derivative works. Users with over 700 million monthly active users need a separate license, and Meta disclaims all warranties and limits liability for any use of the materials.


***
**Coding sources**


* You can run the `meta-llama/Meta-Llama-3-70B-Instruct`, see model page: https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct
    + Hugging Face documentation: https://huggingface.co/docs/transformers/main/en/model_doc/llama3

## Load necessary libraries and data:

In [1]:
import os
import sys

# Assuming 'src' is two levels down (in the current directory or a subdirectory)
path_to_src = os.path.join('..')  # Moves one level down to 'src' folder

# Change the working directory
os.chdir(path_to_src)

# Verify the updated working directory
print("Updated working directory:", os.getcwd())

# load helper functions
import src.API_key as keys

import src.prompt_functions as pf
import src.prompt_text as pt
import src.get_data as get_data

Updated working directory: /home/fenn/Desktop/Publications/basal attributes/Analyses/part_III


load external data:

words of the single partitions and their respective mean_valence:

In [2]:
import pandas as pd

df_partitions = pd.read_excel('./data/LeidenAlgorithm_solution.xlsx')
df_partitions.rename(columns={'parition': 'partition'}, inplace=True)

In [3]:
df_partitions

,partition,words,mean_valence,sd_valence
0,1,aktive Formänderung durch Umwelteinwirkung,0.18,1.140892
1,1,reaktionsfähig,0.67,1.111209
2,1,passive Formänderung durch Umwelteinwirkung,-0.02,1.131126
3,1,autonom,0.75,0.949819
4,1,passive Verhaltensänderung durch Umwelteinwirkung,0.06,1.056184
5,1,aktive Verhaltensänderung durch Umwelteinwirkung,0.27,1.148185
6,1,intelligent,1.35,1.103033
7,1,multifunktional,1.65,1.024702
8,1,technologisch,0.76,0.959081
9,2,zuverlässig,2.17,1.017770


combination of partitions I want to generate a text from:

In [4]:
df_hypothesis= pd.read_excel('./data/LeidenAlgorithm_hypotheses.xlsx')
df_hypothesis.rename(columns={'partitionA': 'partitionA', 'partitionB': 'partitionB'}, inplace=True)


In [5]:
df_hypothesis = df_hypothesis[df_hypothesis['choose'] == 1][['partitionA', 'partitionB']]
df_hypothesis = df_hypothesis.reset_index(drop=True)
df_hypothesis

,partitionA,partitionB
0,3,6
1,3,5
2,1,6
3,1,5
4,1,2
5,2,4


In [6]:
import pandas as pd

# Assuming df_partitions and df_hypothesis are already defined

# Function to compute mean and std for combined partitions
def compute_partition_stats(row):
    partition_a = row['partitionA']
    partition_b = row['partitionB']
    
    # Filter df_partitions for the two partitions
    subset = df_partitions[df_partitions['partition'].isin([partition_a, partition_b])]
    
    # Compute mean and std of mean_valence
    mean_val = subset['mean_valence'].mean()
    std_val = subset['mean_valence'].std()
    
    return pd.Series({'computed_mean': mean_val, 'computed_sd': std_val})

# Apply function to each row in df_hypothesis
df_hypothesis[['computed_mean', 'computed_sd']] = df_hypothesis.apply(compute_partition_stats, axis=1)

df_hypothesis


,partitionA,partitionB,computed_mean,computed_sd
0,3,6,-0.927143,1.288264
1,3,5,-0.093750,1.954942
2,1,6,0.555000,0.545952
3,1,5,0.953846,0.706830
4,1,2,1.220000,0.775879
5,2,4,1.858333,0.516682


## Load the prompt template and model:

prompt template:

In [7]:
from importlib import reload

pt = reload(pt)
print("pt.user_template:\n", pt.user_template)
print("\npt.system_template:\n", pt.system_template)

pt.user_template:
 
<Liste der zu verwendenden basalen Attribute: 
({items_list})>


pt.system_template:
 
<Aufgabe:
Entwickle eine für Laien verständliche, aber wissenschaftlich fundierte Beschreibung eines neuartigen Jackensystems namens Nano-Pat-Parka. 
Ziel ist es, anhand sogenannter basaler Attribute – also grundlegender, semantisch und emotional bewerteter Merkmale neuer Technologien – die möglichen Vorteile und Herausforderungen dieser Technologie zu verdeutlichen.

Basale Attribute sind adjektivische Eigenschaften (z. B. „autonom“, „wartungsintensiv“, „bioinspiriert“), die typischerweise verwendet werden, um neue technologische Systeme zu beschreiben. 
Sie dienen der kognitiven und affektiven Bewertung und ermöglichen eine strukturierte Beschreibung technischer Systeme auf einer allgemeinen Ebene – unabhängig von konkreten Details.

Die Beschreibung des Nano-Pat-Parka soll sich mehrheitliich auf diese basalen Attribute stützen, ohne große Ausführungen zu konkreten technischen F

model:

In [8]:
pf = reload(pf)
pf.huggingface_API_call

<function src.prompt_functions.huggingface_API_call(prompt, items_list, api_key, model_name='meta-llama/llama-3.3-70b-instruct', json_schema=None, max_tokens=1000, temperature=0.0, verbose=True)>

## Run the model:

a non reasoning model is currently not working:

In [9]:
row = df_hypothesis.iloc[0]

# Ensure the value is list-like
partitionA = [row['partitionA']]
partitionB = [row['partitionB']]
combined_partitions = list(partitionA) + list(partitionB)

combined_subset = df_partitions[df_partitions['partition'].isin(combined_partitions)]["words"].values
combined_subset = " // ".join(combined_subset.tolist())
combined_subset

'wartungsintensiv // enthält Kunststoff // leicht zerstörbar // umweltschädlich // Insekten ähnlich // bioinspiriert // lebensähnlich'

a non reasoning model is currently not working:

In [10]:
result = pf.huggingface_API_call(prompt=pt.prompt_template,
                     items_list=combined_subset,
                     api_key=keys.hugging_api_key,
                     model_name="meta-llama/llama-3.3-70b-instruct", temperature=0)

Tokens Used: 630
	Prompt Tokens: 558
	Completion Tokens: 72
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 630
Prompt Tokens: 558
Completion Tokens: 72
Total Cost (USD): $0.0


In [11]:
print(result.content)

word_count = len(result.content.split())
print("Number of words:", word_count)

Die Entwicklung von Schutzkleidung reagiert auf zukünftige Anforderungen. 
Das Nano-Pat-Parka-System ist bioinspiriert und lebensähnlich, aber auch leicht zerstörbar und umweltschädlich. 
Die Technologie ist ein bioinspiriertes System.
Number of words: 26


a reasoning model is working:

In [12]:
result = pf.huggingface_API_call(prompt=pt.prompt_template,
                     items_list=combined_subset,
                     api_key=keys.hugging_api_key,
                     model_name="deepseek/deepseek-r1-turbo", temperature=0)

Tokens Used: 959
	Prompt Tokens: 498
	Completion Tokens: 461
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 959
Prompt Tokens: 498
Completion Tokens: 461
Total Cost (USD): $0.0


In [13]:
import re

def parse_result_content(content):
    # Extract <think>...</think>
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    think_block = think_match.group(1).strip() if think_match else None

    # Extract German description (between </think> and the next ###)
    description_match = re.search(r"</think>\s*\n+(.*?)(?=\n\s*\*)", content, re.DOTALL)
    german_description = description_match.group(1).strip() if description_match else None

    # Extract Wortzahl
    word_count_match = re.search(r"\*Wortzahl:\s*(\d+)\*", content)
    wortzahl = int(word_count_match.group(1)) if word_count_match else None

    # Check if all attributes were integrated
    attributes_confirmed = re.search(r"\*Alle vorgegebenen Attribute integriert\.*\*", content) is not None

    return {
        "think_block": think_block,
        "german_description": german_description,
        "word_count": wortzahl,
        "attributes_confirmed": attributes_confirmed
    }

# Example usage:
parsed_data = parse_result_content(result.content)
print(parsed_data)



print("Number of words by LLM:", parsed_data["word_count"])

word_count = len(parsed_data["german_description"].split())
print("Number of words counted:", word_count)

{'think_block': 'Okay, let\'s tackle this. The user wants a description of the Nano-Pat-Parka using specific attributes. First, I need to make sure I understand all the requirements. The text must be 60-80 words, neutral, third person. Use all the listed attributes without mixing conflicting emotional tones. Start with an intro about the application area and end with a summary.\n\nThe attributes are: wartungsintensiv, enthält Kunststoff, leicht zerstörbar, umweltschädlich, Insekten ähnlich, bioinspiriert, lebensähnlich. Hmm, some of these are negative (umweltschädlich, leicht zerstörbar) and others positive or neutral (bioinspiriert, Insekten ähnlich). Need to present them in a balanced way without contradiction.\n\nStart with the intro: maybe mention future protective clothing. Then integrate each attribute. "Bioinspiriert" and "Insekten ähnlich" can be linked to design inspired by insects. "Lebensähnlich" suggests adaptive features. Then address the negatives: contains plastic, envir

In [14]:
parsed_data["german_description"]

'Das Nano-Pat-Parka-System zielt auf zukünftige Anforderungen an Schutzkleidung durch eine Kombination bioinspirierter und lebensähnlicher Eigenschaften. Die Technologie nutzt insektenähnliche Mikrostrukturen, um atmungsaktive Flexibilität mit Schutzfunktionen zu verbinden. Trotz des Einsatzes von Kunststoffkomponenten ist das Material leicht zerstörbar und erfordert dadurch eine wartungsintensive Pflege. Kritisch bleibt die potenziell umweltschädliche Produktion synthetischer Nanopartikel. Insgesamt vereint die Innovation natürliche Vorbilder mit technischer Anpassungsfähigkeit, steht jedoch vor ökologischen und praktischen Zielkonflikten.'

In [20]:
print(result.content)

<think>
Okay, let's tackle this. The user wants a description of the Nano-Pat-Parka using specific attributes. First, I need to make sure I understand all the requirements. The text must be 60-80 words, neutral, third person. Use all the listed attributes without mixing conflicting emotional tones. Start with an intro about the application area and end with a summary.

The attributes are: wartungsintensiv, enthält Kunststoff, leicht zerstörbar, umweltschädlich, Insekten ähnlich, bioinspiriert, lebensähnlich. Hmm, some of these are negative (umweltschädlich, leicht zerstörbar) and others positive or neutral (bioinspiriert, Insekten ähnlich). Need to present them in a balanced way without contradiction.

Start with the intro: maybe mention future protective clothing. Then integrate each attribute. "Bioinspiriert" and "Insekten ähnlich" can be linked to design inspired by insects. "Lebensähnlich" suggests adaptive features. Then address the negatives: contains plastic, environmentally har

### loop through the partitions and generate a text for each partition:

In [16]:
for index, row in df_hypothesis.iterrows():
    print(f"Index {index}")

    # Ensure the value is list-like
    partitionA = [row['partitionA']]
    partitionB = [row['partitionB']]
    combined_partitions = list(partitionA) + list(partitionB)

    combined_subset = df_partitions[df_partitions['partition'].isin(combined_partitions)]["words"].values
    combined_subset = " // ".join(combined_subset.tolist())

Index 0
Index 1
Index 2
Index 3
Index 4
Index 5


In [17]:
# get_data.save_results_to_csv('result.csv', result)

In [18]:
### implement
#get_data=reload(get_data)
#text_check = get_data.check_for_missing_matching_words(result_df['text'], df_partitions, [3,6], 0.5)
#print(text_check)